In [ ]:
!pip install -q datasets scikit-learn pandas numpy matplotlib seaborn torch

In [ ]:
from datasets import load_dataset

dataset = load_dataset("SetFit/go_emotions")

print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

In [ ]:
for split in dataset:
    print(f"\n{split.upper()}")
    print("Number of rows:", len(dataset[split]))
    print("Columns:", dataset[split].column_names)

In [ ]:
import re
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

label_columns = dataset["train"].column_names[1:]

MAX_VOCAB = 30000
MAX_LEN = 50
BATCH_SIZE = 128

def tokenize(text):
    return re.findall(r"\b\w+\b|[^\w\s]", text.lower())

counter = Counter()

for text in dataset["train"]["text"]:
    counter.update(tokenize(text))

vocab = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Number of emotion labels:", len(label_columns))

def encode_text(text):
    tokens = tokenize(text)[:MAX_LEN]
    ids = [vocab.get(token, 1) for token in tokens]
    ids += [0] * (MAX_LEN - len(ids))
    return ids

class EmotionDataset(Dataset):
    def __init__(self, split):
        data = dataset[split]
        self.X = torch.tensor(
            [encode_text(text) for text in data["text"]],
            dtype=torch.long
        )
        self.y = torch.tensor(
            np.array(data.select_columns(label_columns).to_pandas()),
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

train_data = EmotionDataset("train")
val_data = EmotionDataset("validation")
test_data = EmotionDataset("test")

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    val_data, batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    test_data, batch_size=BATCH_SIZE, shuffle=False
)

print("Training shape:", train_data.X.shape, train_data.y.shape)
print("Validation shape:", val_data.X.shape, val_data.y.shape)
print("Testing shape:", test_data.X.shape, test_data.y.shape)

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class EmotionMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, 28)

    def forward(self, x):
        embeddings = self.embedding(x)

        mask = (x != 0).unsqueeze(-1)
        embeddings = embeddings * mask

        pooled = embeddings.sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        hidden = self.fc1(pooled)
        hidden = self.relu(hidden)
        hidden = self.dropout(hidden)

        return self.fc2(hidden)

model = EmotionMLP(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Device: cpu
EmotionMLP(
  (embedding): Embedding(26774, 100, padding_idx=0)
  (fc1): Linear(in_features=100, out_features=128, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=28, bias=True)
)
Total parameters: 2693940


In [28]:
from sklearn.metrics import f1_score
import copy

EPOCHS = 5
best_val_loss = float("inf")
best_state = None

history = {
    "train_loss": [],
    "val_loss": [],
    "val_f1": []
}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            val_loss += loss.item() * X.size(0)
            all_probs.append(torch.sigmoid(outputs).cpu())
            all_labels.append(y.cpu())

    val_loss /= len(val_loader.dataset)

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    predictions = (probs >= 0.5).astype(int)

    val_f1 = f1_score(
        labels, predictions, average="micro", zero_division=0
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Micro-F1: {val_f1:.4f}"
    )

model.load_state_dict(best_state)
torch.save(model.state_dict(), "baseline_ann.pth")

print("Best baseline model saved.")

Epoch 1/5 | Train Loss: 0.6628 | Val Loss: 0.6368 | Val Micro-F1: 0.1037
Epoch 2/5 | Train Loss: 0.6116 | Val Loss: 0.5826 | Val Micro-F1: 0.0436
Epoch 3/5 | Train Loss: 0.5530 | Val Loss: 0.5174 | Val Micro-F1: 0.0105
Epoch 4/5 | Train Loss: 0.4819 | Val Loss: 0.4400 | Val Micro-F1: 0.0022
Epoch 5/5 | Train Loss: 0.4029 | Val Loss: 0.3607 | Val Micro-F1: 0.0006
Best baseline model saved.


In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        outputs = model(X)

        all_probs.append(torch.sigmoid(outputs).cpu())
        all_labels.append(y)

probs = torch.cat(all_probs).numpy()
y_true = torch.cat(all_labels).numpy()
y_pred = (probs >= 0.5).astype(int)

print("Exact-match accuracy:",
      accuracy_score(y_true, y_pred))
print("Micro precision:",
      precision_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro recall:",
      recall_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro F1:",
      f1_score(y_true, y_pred, average="micro", zero_division=0))
print("Macro F1:",
      f1_score(y_true, y_pred, average="macro", zero_division=0))

Exact-match accuracy: 0.0
Micro precision: 0.0
Micro recall: 0.0
Micro F1: 0.0
Macro F1: 0.0


In [30]:
print("True positive labels:", y_true.sum())
print("Predicted positive labels:", y_pred.sum())
print("Average prediction probability:", probs.mean())
print("Maximum prediction probability:", probs.max())
print("Minimum prediction probability:", probs.min())

print("\nTrue labels for first 5 comments:")
print(y_true[:5])

print("\nPredicted probabilities for first 5 comments:")
print(probs[:5])

True positive labels: 6329.0
Predicted positive labels: 3
Average prediction probability: 0.27800286
Maximum prediction probability: 0.50429535
Minimum prediction probability: 0.06139388

True labels for first 5 comments:
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 1.]]

Predicted probabilities for first 5 comments:
[[0.28577292 0.32082957 0.27212343 0.32128194 0.36132511 0.26536068
  0.31623653 0.320539   0.2879349  0.29116982 0.27496353 0.23844382
  0.2730808  0.31829152 0.29633388 0.31807545 0.24138176 0.29066792
  0.27654117 0.3014551  0.2903426  0.2948422  0.28463608 0.2646795
  0.26564643 0

In [31]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

model = EmotionMLP(len(vocab)).to(device)

positive = train_data.y.sum(dim=0)
negative = len(train_data) - positive

pos_weight = (negative / positive.clamp(min=1)).clamp(max=10).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.SGD(
    model.parameters(),
    lr=0.05,
    momentum=0.9
)

EPOCHS = 15
best_val_f1 = -1
best_state = None

history = {
    "train_loss": [],
    "val_loss": [],
    "val_f1": []
}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

    train_loss = total_loss / len(train_data)

    model.eval()
    val_loss = 0
    val_probs = []
    val_labels = []

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            val_loss += loss.item() * X.size(0)
            val_probs.append(torch.sigmoid(outputs).cpu())
            val_labels.append(y.cpu())

    val_loss /= len(val_data)

    probs_val = torch.cat(val_probs).numpy()
    labels_val = torch.cat(val_labels).numpy()

    pred_val = (probs_val >= 0.5).astype(int)

    val_f1 = f1_score(
        labels_val,
        pred_val,
        average="micro",
        zero_division=0
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Micro-F1: {val_f1:.4f}"
    )

model.load_state_dict(best_state)
torch.save(model.state_dict(), "baseline_ann.pth")

print("Best validation micro-F1:", best_val_f1)
print("Model saved successfully.")

Epoch 1/15 | Train Loss: 0.6771 | Val Loss: 0.6284 | Val Micro-F1: 0.2809
Epoch 2/15 | Train Loss: 0.6136 | Val Loss: 0.5968 | Val Micro-F1: 0.3196
Epoch 3/15 | Train Loss: 0.5908 | Val Loss: 0.5798 | Val Micro-F1: 0.3276
Epoch 4/15 | Train Loss: 0.5768 | Val Loss: 0.5678 | Val Micro-F1: 0.3410
Epoch 5/15 | Train Loss: 0.5667 | Val Loss: 0.5595 | Val Micro-F1: 0.3353
Epoch 6/15 | Train Loss: 0.5593 | Val Loss: 0.5535 | Val Micro-F1: 0.3365
Epoch 7/15 | Train Loss: 0.5531 | Val Loss: 0.5480 | Val Micro-F1: 0.3458
Epoch 8/15 | Train Loss: 0.5480 | Val Loss: 0.5438 | Val Micro-F1: 0.3464
Epoch 9/15 | Train Loss: 0.5431 | Val Loss: 0.5401 | Val Micro-F1: 0.3501
Epoch 10/15 | Train Loss: 0.5389 | Val Loss: 0.5370 | Val Micro-F1: 0.3452
Epoch 11/15 | Train Loss: 0.5349 | Val Loss: 0.5342 | Val Micro-F1: 0.3467
Epoch 12/15 | Train Loss: 0.5316 | Val Loss: 0.5317 | Val Micro-F1: 0.3527
Epoch 13/15 | Train Loss: 0.5283 | Val Loss: 0.5285 | Val Micro-F1: 0.3502
Epoch 14/15 | Train Loss: 0.5254 |

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

model.eval()
val_probs = []
val_labels = []

with torch.no_grad():
    for X, y in val_loader:
        outputs = model(X.to(device))
        val_probs.append(torch.sigmoid(outputs).cpu())
        val_labels.append(y)

val_probs = torch.cat(val_probs).numpy()
val_labels = torch.cat(val_labels).numpy()

thresholds = np.arange(0.10, 0.56, 0.05)
scores = []

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(int)

    score = f1_score(
        val_labels,
        predictions,
        average="micro",
        zero_division=0
    )

    scores.append(score)
    print(f"Threshold: {threshold:.2f} | Validation Micro-F1: {score:.4f}")

best_threshold = float(thresholds[np.argmax(scores)])
print("\nBest validation threshold:", best_threshold)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

model.eval()
test_probs = []
test_labels = []

with torch.no_grad():
    for X, y in test_loader:
        outputs = model(X.to(device))
        test_probs.append(torch.sigmoid(outputs).cpu())
        test_labels.append(y)

test_probs = torch.cat(test_probs).numpy()
y_true = torch.cat(test_labels).numpy()
y_pred = (test_probs >= best_threshold).astype(int)

print("Exact-match accuracy:",
      accuracy_score(y_true, y_pred))
print("Micro precision:",
      precision_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro recall:",
      recall_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro F1:",
      f1_score(y_true, y_pred, average="micro", zero_division=0))
print("Macro F1:",
      f1_score(y_true, y_pred, average="macro", zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Training Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ANN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["val_f1"], marker="o", label="Validation Micro-F1")
plt.xlabel("Epoch")
plt.ylabel("Micro F1-score")
plt.title("Validation Performance Across Epochs")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
class ProposedEmotionMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        self.fc1 = nn.Linear(embed_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 28)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        embeddings = self.embedding(x)
        mask = (x != 0).unsqueeze(-1)

        embeddings = embeddings * mask
        pooled = embeddings.sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        hidden = self.dropout(self.relu(self.fc1(pooled)))
        hidden = self.dropout(self.relu(self.fc2(hidden)))

        return self.fc3(hidden)

proposed_model = ProposedEmotionMLP(len(vocab)).to(device)

print(proposed_model)

In [36]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.SGD(
    proposed_model.parameters(),
    lr=0.05,
    momentum=0.9
)

EPOCHS = 15
best_val_f1 = -1
best_state = None

proposed_history = {
    "train_loss": [],
    "val_loss": [],
    "val_f1": []
}

for epoch in range(EPOCHS):
    proposed_model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = proposed_model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

    train_loss = total_loss / len(train_data)

    proposed_model.eval()
    val_loss = 0
    val_probs = []
    val_labels = []

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)

            outputs = proposed_model(X)
            loss = criterion(outputs, y)

            val_loss += loss.item() * X.size(0)
            val_probs.append(torch.sigmoid(outputs).cpu())
            val_labels.append(y.cpu())

    val_loss /= len(val_data)

    probs_val = torch.cat(val_probs).numpy()
    labels_val = torch.cat(val_labels).numpy()
    pred_val = (probs_val >= 0.5).astype(int)

    val_f1 = f1_score(
        labels_val,
        pred_val,
        average="micro",
        zero_division=0
    )

    proposed_history["train_loss"].append(train_loss)
    proposed_history["val_loss"].append(val_loss)
    proposed_history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(proposed_model.state_dict())

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Micro-F1: {val_f1:.4f}"
    )

proposed_model.load_state_dict(best_state)
torch.save(proposed_model.state_dict(), "proposed_ann.pth")

print("Best validation Micro-F1:", best_val_f1)
print("Proposed ANN saved successfully.")

Epoch 1/15 | Train Loss: 0.6845 | Val Loss: 0.6414 | Val Micro-F1: 0.2717
Epoch 2/15 | Train Loss: 0.6276 | Val Loss: 0.6011 | Val Micro-F1: 0.3222
Epoch 3/15 | Train Loss: 0.5973 | Val Loss: 0.5781 | Val Micro-F1: 0.3212
Epoch 4/15 | Train Loss: 0.5814 | Val Loss: 0.5647 | Val Micro-F1: 0.3402
Epoch 5/15 | Train Loss: 0.5702 | Val Loss: 0.5548 | Val Micro-F1: 0.3520
Epoch 6/15 | Train Loss: 0.5611 | Val Loss: 0.5459 | Val Micro-F1: 0.3447
Epoch 7/15 | Train Loss: 0.5537 | Val Loss: 0.5393 | Val Micro-F1: 0.3519
Epoch 8/15 | Train Loss: 0.5471 | Val Loss: 0.5340 | Val Micro-F1: 0.3562
Epoch 9/15 | Train Loss: 0.5406 | Val Loss: 0.5282 | Val Micro-F1: 0.3674
Epoch 10/15 | Train Loss: 0.5350 | Val Loss: 0.5246 | Val Micro-F1: 0.3602
Epoch 11/15 | Train Loss: 0.5313 | Val Loss: 0.5219 | Val Micro-F1: 0.3676
Epoch 12/15 | Train Loss: 0.5258 | Val Loss: 0.5193 | Val Micro-F1: 0.3749
Epoch 13/15 | Train Loss: 0.5222 | Val Loss: 0.5153 | Val Micro-F1: 0.3730
Epoch 14/15 | Train Loss: 0.5193 |

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

proposed_model.eval()
val_probs = []
val_labels = []

with torch.no_grad():
    for X, y in val_loader:
        outputs = proposed_model(X.to(device))
        val_probs.append(torch.sigmoid(outputs).cpu())
        val_labels.append(y)

val_probs = torch.cat(val_probs).numpy()
val_labels = torch.cat(val_labels).numpy()

thresholds = np.arange(0.10, 0.56, 0.05)
scores = []

for threshold in thresholds:
    pred = (val_probs >= threshold).astype(int)
    score = f1_score(
        val_labels, pred, average="micro", zero_division=0
    )
    scores.append(score)
    print(f"Threshold: {threshold:.2f} | Micro-F1: {score:.4f}")

best_threshold_proposed = float(thresholds[np.argmax(scores)])
print("Selected threshold:", best_threshold_proposed)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

proposed_model.eval()
test_probs = []
test_labels = []

with torch.no_grad():
    for X, y in test_loader:
        outputs = proposed_model(X.to(device))
        test_probs.append(torch.sigmoid(outputs).cpu())
        test_labels.append(y)

test_probs = torch.cat(test_probs).numpy()
y_true_proposed = torch.cat(test_labels).numpy()
y_pred_proposed = (
    test_probs >= best_threshold_proposed
).astype(int)

proposed_results = {
    "Exact-match accuracy": accuracy_score(y_true_proposed, y_pred_proposed),
    "Micro precision": precision_score(y_true_proposed, y_pred_proposed, average="micro", zero_division=0),
    "Micro recall": recall_score(y_true_proposed, y_pred_proposed, average="micro", zero_division=0),
    "Micro F1": f1_score(y_true_proposed, y_pred_proposed, average="micro", zero_division=0),
    "Macro F1": f1_score(y_true_proposed, y_pred_proposed, average="macro", zero_division=0)
}

for metric, value in proposed_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {
        "Model": "Baseline ANN",
        "Exact-match accuracy": 0.181684,
        "Micro precision": 0.347550,
        "Micro recall": 0.372097,
        "Micro F1": 0.359405,
        "Macro F1": 0.203516
    },
    {
        "Model": "Proposed ANN",
        "Exact-match accuracy": 0.2001,
        "Micro precision": 0.3706,
        "Micro recall": 0.4181,
        "Micro F1": 0.3929,
        "Macro F1": 0.2152
    }
])

print(results.round(4).to_string(index=False))
results.to_csv("ann_model_comparison.csv", index=False)

In [ ]:
import copy
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

learning_rates = [0.01, 0.05, 0.1]
lr_results = []

for lr in learning_rates:
    experiment_model = ProposedEmotionMLP(len(vocab)).to(device)

    optimizer = optim.SGD(
        experiment_model.parameters(),
        lr=lr,
        momentum=0.9
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_f1 = -1
    best_epoch = 0
    best_state = None

    for epoch in range(10):
        experiment_model.train()

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = experiment_model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

        experiment_model.eval()
        val_probs = []
        val_labels = []

        with torch.no_grad():
            for X, y in val_loader:
                outputs = experiment_model(X.to(device))
                val_probs.append(torch.sigmoid(outputs).cpu())
                val_labels.append(y)

        val_probs = torch.cat(val_probs).numpy()
        val_labels = torch.cat(val_labels).numpy()

        predictions = (val_probs >= 0.5).astype(int)
        score = f1_score(
            val_labels, predictions,
            average="micro", zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_epoch = epoch + 1
            best_state = copy.deepcopy(experiment_model.state_dict())

    lr_results.append({
        "Learning Rate": lr,
        "Best Epoch": best_epoch,
        "Validation Micro-F1": best_f1
    })

    torch.save(best_state, f"proposed_lr_{lr}.pth")

lr_table = pd.DataFrame(lr_results)
print(lr_table.round(4).to_string(index=False))

In [ ]:
import copy
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

batch_sizes = [32, 64, 128]
batch_results = []

for batch_size in batch_sizes:
    train_exp = DataLoader(
        train_data, batch_size=batch_size, shuffle=True
    )
    val_exp = DataLoader(
        val_data, batch_size=batch_size, shuffle=False
    )

    experiment_model = ProposedEmotionMLP(len(vocab)).to(device)

    optimizer = optim.SGD(
        experiment_model.parameters(),
        lr=0.05,
        momentum=0.9
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_f1 = -1
    best_epoch = 0
    best_state = None

    for epoch in range(10):
        experiment_model.train()

        for X, y in train_exp:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = experiment_model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

        experiment_model.eval()
        val_probs = []
        val_labels = []

        with torch.no_grad():
            for X, y in val_exp:
                outputs = experiment_model(X.to(device))
                val_probs.append(torch.sigmoid(outputs).cpu())
                val_labels.append(y)

        probs_val = torch.cat(val_probs).numpy()
        labels_val = torch.cat(val_labels).numpy()

        pred_val = (probs_val >= 0.5).astype(int)

        score = f1_score(
            labels_val, pred_val,
            average="micro", zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_epoch = epoch + 1
            best_state = copy.deepcopy(experiment_model.state_dict())

    torch.save(best_state, f"proposed_batch_{batch_size}.pth")

    batch_results.append({
        "Batch Size": batch_size,
        "Best Epoch": best_epoch,
        "Validation Micro-F1": best_f1
    })

batch_table = pd.DataFrame(batch_results)
print(batch_table.round(4).to_string(index=False))

 Batch Size  Best Epoch  Validation Micro-F1
         32          10               0.3919
         64          10               0.3787
        128          10               0.3793


In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

torch.manual_seed(42)

train_final = DataLoader(train_data, batch_size=32, shuffle=True)
val_final = DataLoader(val_data, batch_size=32, shuffle=False)
test_final = DataLoader(test_data, batch_size=32, shuffle=False)

final_model = ProposedEmotionMLP(len(vocab)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.SGD(final_model.parameters(), lr=0.05, momentum=0.9)

best_f1 = -1
best_state = None
wait = 0
patience = 3
final_history = []

for epoch in range(15):
    final_model.train()
    total_loss = 0

    for X, y in train_final:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)

    final_model.eval()
    vp, vy = [], []

    with torch.no_grad():
        for X, y in val_final:
            vp.append(torch.sigmoid(final_model(X.to(device))).cpu())
            vy.append(y)

    vp = torch.cat(vp).numpy()
    vy = torch.cat(vy).numpy()
    score = f1_score(vy, vp >= 0.5, average="micro", zero_division=0)

    final_history.append({
        "epoch": epoch + 1,
        "train_loss": total_loss / len(train_data),
        "val_micro_f1": score
    })

    print(f"Epoch {epoch+1}: Train Loss={total_loss/len(train_data):.4f}, Val Micro-F1={score:.4f}")

    if score > best_f1:
        best_f1 = score
        best_state = copy.deepcopy(final_model.state_dict())
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        break

final_model.load_state_dict(best_state)

torch.save(final_model.state_dict(), "final_ann_model.pth")
pd.DataFrame(final_history).to_csv("final_training_history.csv", index=False)

vp, vy = [], []

with torch.no_grad():
    for X, y in val_final:
        vp.append(torch.sigmoid(final_model(X.to(device))).cpu())
        vy.append(y)

vp = torch.cat(vp).numpy()
vy = torch.cat(vy).numpy()

thresholds = np.arange(0.10, 0.56, 0.05)
threshold_scores = [
    f1_score(vy, vp >= t, average="micro", zero_division=0)
    for t in thresholds
]
best_threshold = float(thresholds[np.argmax(threshold_scores)])
print("Selected validation threshold:", best_threshold)

tp, ty = [], []

with torch.no_grad():
    for X, y in test_final:
        tp.append(torch.sigmoid(final_model(X.to(device))).cpu())
        ty.append(y)

tp = torch.cat(tp).numpy()
ty = torch.cat(ty).numpy()
pred = (tp >= best_threshold).astype(int)

final_results = {
    "Exact-match accuracy": accuracy_score(ty, pred),
    "Micro precision": precision_score(ty, pred, average="micro", zero_division=0),
    "Micro recall": recall_score(ty, pred, average="micro", zero_division=0),
    "Micro F1": f1_score(ty, pred, average="micro", zero_division=0),
    "Macro F1": f1_score(ty, pred, average="macro", zero_division=0)
}

print("\nFINAL TEST RESULTS")
for name, value in final_results.items():
    print(f"{name}: {value:.4f}")

pd.DataFrame([final_results]).to_csv("final_test_results.csv", index=False)
print("Saved final_ann_model.pth, final_training_history.csv, final_test_results.csv")

In [41]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import zipfile
import os
from google.colab import files

os.makedirs("ANN_Graphs", exist_ok=True)

def save_graph(filename):
    plt.tight_layout()
    plt.savefig(f"ANN_Graphs/{filename}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 1. Training and validation loss
if "final_history" in globals() and len(final_history) > 0:
    h = pd.DataFrame(final_history)

    plt.figure(figsize=(8, 5))
    plt.plot(h["epoch"], h["train_loss"], marker="o", label="Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.legend()
    plt.grid(True)
    save_graph("01_training_loss.png")

    plt.figure(figsize=(8, 5))
    plt.plot(h["epoch"], h["val_micro_f1"], marker="o", label="Validation Micro-F1")
    plt.xlabel("Epoch")
    plt.ylabel("Micro-F1")
    plt.title("Validation Micro-F1 Across Epochs")
    plt.legend()
    plt.grid(True)
    save_graph("02_validation_micro_f1.png")

if "history" in globals() and len(history.get("train_loss", [])) > 0:
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Training Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Baseline Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    save_graph("03_baseline_loss_comparison.png")

# 2. Learning-rate experiment
if "lr_table" in globals():
    plt.figure(figsize=(8, 5))
    plt.bar(
        lr_table["Learning Rate"].astype(str),
        lr_table["Validation Micro-F1"]
    )
    plt.xlabel("Learning Rate")
    plt.ylabel("Validation Micro-F1")
    plt.title("Learning Rate Comparison")
    plt.grid(axis="y")
    save_graph("04_learning_rate_comparison.png")

# 3. Batch-size experiment
if "batch_table" in globals():
    plt.figure(figsize=(8, 5))
    plt.bar(
        batch_table["Batch Size"].astype(str),
        batch_table["Validation Micro-F1"]
    )
    plt.xlabel("Batch Size")
    plt.ylabel("Validation Micro-F1")
    plt.title("Batch Size Comparison")
    plt.grid(axis="y")
    save_graph("05_batch_size_comparison.png")

# 4. Baseline vs proposed model
if "results" in globals():
    metric_cols = [
        "Exact-match accuracy",
        "Micro precision",
        "Micro recall",
        "Micro F1",
        "Macro F1"
    ]

    plot_data = results.set_index("Model")[metric_cols]

    plot_data.T.plot(kind="bar", figsize=(11, 6))
    plt.ylabel("Score")
    plt.title("Baseline vs Proposed ANN")
    plt.xticks(rotation=25, ha="right")
    plt.legend(title="Model")
    plt.grid(axis="y")
    save_graph("06_baseline_vs_proposed.png")

# 5. Final test metrics
if "final_results" in globals():
    plt.figure(figsize=(9, 5))
    plt.bar(
        list(final_results.keys()),
        list(final_results.values())
    )
    plt.ylabel("Score")
    plt.title("Final ANN Test Metrics")
    plt.xticks(rotation=25, ha="right")
    plt.ylim(0, 1)
    plt.grid(axis="y")
    save_graph("07_final_test_metrics.png")

# Zip all generated graphs
with zipfile.ZipFile("ANN_All_Graphs.zip", "w") as z:
    for filename in os.listdir("ANN_Graphs"):
        if filename.endswith(".png"):
            z.write(
                os.path.join("ANN_Graphs", filename),
                arcname=filename
            )

print("Graphs generated:", os.listdir("ANN_Graphs"))
files.download("ANN_All_Graphs.zip")

Graphs generated: ['04_learning_rate_comparison.png.png', '05_batch_size_comparison.png.png', '03_baseline_loss_comparison.png.png', '06_baseline_vs_proposed.png.png']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>